In [3]:
%run DNCfundamentals.ipynb

In [ ]:
# Input: a valid leading partition (i.e. non hook partition)
# Output: the uniqute caterpillar with that leading partition that is part of the caterpillar-basis
#         introduced in the paper by Gonzalez-Orellana-Tomba
def basis_caterpillar(leading):
    G = Graph()
    internal_vertices = [0]
    
    if len(leading) == 1:
        G = G.disjoint_union(graphs.StarGraph(leading[0]-1), labels='integers')
    if leading[1] == 1:
        return None
    else:
        for i in range(1,len(leading)-1):
            internal_vertices.append(leading[i]+internal_vertices[i-1])

        internal_vertices.append(internal_vertices[-1]+leading[-1])

        for i in range(1,len(leading)):
            G = G.disjoint_union(graphs.StarGraph(leading[i]-1), labels='integers')
            
        G = G.disjoint_union(graphs.StarGraph(leading[0]-1), labels='integers')

        for i in range(0, len(internal_vertices)-1):
            G.add_edge([internal_vertices[i],internal_vertices[i+1]])
        # G.plot().show()
    return G

# Input: an integer n
# Output: a list containing all non-hook partitions of n
    
def leading_partition_list(n):
    partitions_list = Partitions(n).list()
    leadings_list = []
    
    for p in partitions_list:
        if len(p) == 1:
            leadings_list.append(p)
        else:
            if p[1] != 1:
                leadings_list.append(p)
                
    return leadings_list

# Ouput: matrix in which each row contains the CSF vector of a caterpillar in the caterpillar basis
#        introduce by Gonzalez-Orellana-Tomba
def basis_matrix(n):
    tree_list = []
    partitions_list = Partitions(n).list()
    leadings_list = leading_partition_list(n)
    
    seen_list = {}
    
    L_P = len(partitions_list)
    M = np.zeros((0, L_P))
    
    for partition in leadings_list:
        T = basis_caterpillar(partition)
        tree_list.append(T)
        
    for i in range(0,len(tree_list)):
        tree_list[i] = tree_list[i].copy()
        # tree_list[i].plot().show()
        CSF_vector = CSF_helper(tree_list[i], len(tree_list[i].vertices()), seen_list)
        # print(CSF_vector)
        M = np.vstack((M, CSF_vector))
    return M

# Input: basismatrix is the CSF matrix where CSF vectors are written in rows
# Input: CSFvector is the CSF of the tree whose linear combination we are trying to find
# Output: vector of coefficients that identify the linear combination

def find_basis_coefficients(basismatrix, csfvector):
    np.set_printoptions(suppress=True)
    # csfvector_transpose = csfvector.tranpose() # gives the csf vector as a column vector
    basismatrix_transpose = basismatrix.transpose() # this gives the CSF basis matrix with vectors in columns
    
    basis_mult = np.matmul(basismatrix, basismatrix_transpose)
    # print(basis_mult)
    
    rhs_mult = np.matmul(basismatrix, csfvector)
    # print(rhs_mult)
    
    basis_mult_inverse = np.linalg.inv(basis_mult)
    # print(basis_mult_inverse)
    
    coefficients = np.matmul(basis_mult_inverse, rhs_mult)
    
    return coefficients